# Auditoría de experimentos — Sentiment140 (Laboratorio II)

Apoyo para la sustentación (sección 8 de la guía). Al ejecutarse contra el Tracking Server de MLflow,
reconstruye el protocolo experimental, los runs presentados y la contribución de cada integrante —
no forma parte del contrato de caja negra, pero debe permitir verificar manualmente lo que audita
la API (`/audit/protocol`, `/audit/runs`, `/audit/contributions`, `/audit/model`).

Incluye además una sección de análisis exploratorio sobre los embeddings de `R_SPACY`.

In [ ]:
import sys
sys.path.insert(0, "..")

import mlflow
import pandas as pd

MLFLOW_TRACKING_URI = "http://3.208.78.52:5000"
EXPERIMENT_NAME = "nlp-lab2-sentiment140"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name(EXPERIMENT_NAME)
print(f"Experiment: {EXPERIMENT_NAME} ({exp.experiment_id})")

## 1. Protocolo experimental

Reconstruye el run único con `lab_run_type=protocol` — debe existir exactamente uno (Anexo A.2).

In [ ]:
protocol_runs = client.search_runs([exp.experiment_id], filter_string="tags.lab_run_type = 'protocol'")
assert len(protocol_runs) == 1, f"Se esperaba exactamente 1 run de protocolo, hay {len(protocol_runs)}"
protocol_run = protocol_runs[0]

print("protocol_run_id:", protocol_run.info.run_id)
print("status:", protocol_run.info.status)
for k, v in protocol_run.data.params.items():
    print(f"  {k} = {v}")
artifacts = client.list_artifacts(protocol_run.info.run_id, "protocol")
print("artifacts:", [a.path for a in artifacts])

## 2. Runs presentados

Todo run con tag `lab_run_type` en `{protocol, experiment, final}` cuenta como presentado.

In [ ]:
all_runs = client.search_runs([exp.experiment_id], filter_string="tags.lab_run_type != ''", max_results=1000)
rows = []
for r in all_runs:
    rows.append({
        "run_id": r.info.run_id,
        "run_type": r.data.tags.get("lab_run_type"),
        "experiment_id_tag": r.data.tags.get("lab_experiment_id"),
        "configuration_id": r.data.tags.get("lab_configuration_id"),
        "stage": r.data.tags.get("lab_stage"),
        "member_id": r.data.tags.get("lab_member_id"),
        "status": r.info.status,
        "macro_f1_mean": r.data.metrics.get("macro_f1_mean"),
        "macro_f1_std": r.data.metrics.get("macro_f1_std"),
    })
runs_df = pd.DataFrame(rows).sort_values("run_id").reset_index(drop=True)
runs_df

## 3. Contribución por integrante

Runs experimentales válidos (`FINISHED`, con `lab_member_id`) agrupados por integrante — mínimo exigido:
`valid_configurations >= 3` en al menos 2 etapas distintas (T0/B0 no cuentan).

In [ ]:
# El minimo individual (A.5 /audit/contributions) se cuenta por lab_configuration_id,
# NO por lab_experiment_id: las comparaciones EXTRA comparten el mismo lab_experiment_id
# ("EXTRA") pero cada una es una configuracion distinta (lab_configuration_id distinto).
exp_runs = runs_df[(runs_df["run_type"] == "experiment") & (runs_df["status"] == "FINISHED")]
countable = exp_runs[~exp_runs["experiment_id_tag"].isin(["T0", "B0"])]

contrib = countable.groupby("member_id").agg(
    valid_configurations=("configuration_id", "nunique"),
    stages=("stage", lambda s: sorted(set(s))),
).reset_index()
contrib["cumple_minimo"] = contrib.apply(
    lambda row: row["valid_configurations"] >= 3 and len(row["stages"]) >= 2, axis=1
)
contrib

## 4. Modelo final registrado

Verifica `sentiment140@champion` en el Model Registry (si ya existe).

In [ ]:
MODEL_NAME = "sentiment140"
MODEL_ALIAS = "champion"

try:
    mv = client.get_model_version_by_alias(MODEL_NAME, MODEL_ALIAS)
    print(f"{MODEL_NAME}@{MODEL_ALIAS} -> version {mv.version}, run_id={mv.run_id}")
except mlflow.exceptions.MlflowException as e:
    print(f"Todavía no existe {MODEL_NAME}@{MODEL_ALIAS}: {e}")

## 5. Análisis exploratorio — embeddings de `R_SPACY` en 2D

Antes de comprometernos con `R_SPACY` como representación en la etapa de "Representación" (sección 3
de la guía), vale la pena mirar cómo se ven los embeddings preentrenados de spaCy en el espacio de
sentimiento — sin entrenar ningún clasificador todavía, solo como inspección visual.

**Qué es el embedding.** `R_SPACY` usa `en_core_web_md` (vectores preentrenados de 300 dimensiones por
palabra) y define el vector de un documento completo como el **promedio** de los vectores de sus tokens
(excluyendo stopwords y puntuación) — esto es exactamente `document_vector_method=mean_token_vectors`
en `pipeline/config.py::config_r_spacy`. Cada tweet queda representado por un único punto en un espacio
de 300 dimensiones.

**Por qué reducir a 2D.** 300 dimensiones no se pueden visualizar directamente. Usamos **PCA**
(Análisis de Componentes Principales) para proyectar cada vector de 300D a solo 2 números, eligiendo
las 2 direcciones del espacio original que capturan la mayor variación entre documentos:

- **Eje X = PC1 (primera componente principal):** la única dirección lineal en el espacio de 300D que,
  por sí sola, explica la mayor cantidad de varianza entre los documentos de la muestra.
- **Eje Y = PC2 (segunda componente principal):** la siguiente dirección de mayor varianza,
  perpendicular (ortogonal) a PC1 — captura la varianza que PC1 no alcanzó a explicar.
- **Ambos ejes son combinaciones lineales abstractas** de las 300 dimensiones originales del embedding;
  no tienen una unidad ni un significado semántico directo (no es "eje de positividad" ni nada por el
  estilo) — lo único interpretable es la *estructura* que forman los puntos.
- **Color/forma de cada punto = la etiqueta real** (`negative` en rojo, `positive` en verde/triángulo)
  del tweet correspondiente — se usa solo para *colorear después* de proyectar, el PCA en sí no conoce
  las etiquetas (es no supervisado).

**Cómo leerlo.** Si el sentimiento fuera la fuente dominante de varianza léxica en el corpus, veríamos
dos nubes de puntos bien separadas por color. Si en cambio los colores están mezclados (como es el caso
aquí — ver más abajo), significa que en las primeras 2 componentes principales, la variación de
*vocabulario/tema* domina sobre la variación de *polaridad*. Esto **no implica que `R_SPACY` vaya a
rendir mal** como representación para Logistic Regression: un clasificador lineal puede aprovechar
información de sentimiento que vive en combinaciones de las otras ~298 dimensiones que PCA no prioriza
aquí. Es una foto exploratoria de dos dimensiones, no un techo de rendimiento.

In [ ]:
import numpy as np
import spacy
from sklearn.decomposition import PCA

from pipeline.data import load_train_test

N_SAMPLE = 1500  # submuestra rápida de exploración, no un run oficial

train_split, _ = load_train_test()
train_df = train_split.to_pandas().reset_index().rename(columns={"index": "index"})
partitions = pd.read_csv("../protocol/partitions.csv")
sample = partitions.merge(train_df[["index", "text", "label"]], on="index", how="left")
sub = sample.sample(n=N_SAMPLE, random_state=42).reset_index(drop=True)

nlp = spacy.load("en_core_web_md", exclude=["parser", "ner"])
vectors = []
for doc in nlp.pipe(sub["text"].tolist(), batch_size=256):
    toks = [t.vector for t in doc if t.has_vector and not t.is_stop and not t.is_punct]
    vectors.append(np.mean(toks, axis=0) if toks else np.zeros(nlp.vocab.vectors_length))
X = np.vstack(vectors)

pca = PCA(n_components=2, random_state=42)
X2d = pca.fit_transform(X)
print(f"Varianza explicada: PC1={pca.explained_variance_ratio_[0]:.1%} PC2={pca.explained_variance_ratio_[1]:.1%} (total={sum(pca.explained_variance_ratio_):.1%})")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
for label, color, marker in [(0, "#e34948", "o"), (1, "#1baf7a", "^")]:
    mask = sub["label"].values == label
    ax.scatter(X2d[mask, 0], X2d[mask, 1], c=color, marker=marker, s=18, alpha=0.6,
               label="negative" if label == 0 else "positive")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} de varianza)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} de varianza)")
ax.set_title("Embeddings de R_SPACY (en_core_web_md, mean_token_vectors) proyectados a 2D con PCA")
ax.legend()
plt.tight_layout()
plt.show()

**Lectura de este resultado concreto:** PC1 y PC2 juntas explican solo una fracción modesta de la
varianza total (impresa arriba) — con 300 dimensiones originales, es normal que las dos primeras
componentes capturen relativamente poco. Las clases `negative`/`positive` aparecen mayormente
entremezcladas en este plano 2D, sin una frontera visual limpia. Esto es información exploratoria
para la discusión de equipo al decidir entre `R_BOW`, `R_TFIDF_UNI`, `R_TFIDF_UNI_BI` y `R_SPACY`
en la etapa de representación — no reemplaza la comparación real por validación cruzada exigida
en la sección 3 de la guía, que es la que efectivamente decide.